In [2]:
import pandas as pd
import numpy as np


In [3]:
# Load purchases (you already did this in 01 notebook, but we repeat in 02)
purchases = pd.read_csv("../data/purchases.csv")

# Convert InvoiceDate to datetime and extract Year
purchases['InvoiceDate'] = pd.to_datetime(purchases['InvoiceDate'])
purchases['Year'] = purchases['InvoiceDate'].dt.year

# Keep only what we need for this step
purch_cols = ['VendorNumber', 'VendorName', 'Year', 'Quantity', 'Dollars']
purch = purchases[purch_cols].copy()

# Aggregate to VendorNumber + VendorName + Year
purch_agg = (
    purch
    .groupby(['VendorNumber', 'VendorName', 'Year'], as_index=False)
    .agg(
        PurchaseQty=('Quantity', 'sum'),
        PurchaseDollars=('Dollars', 'sum')
    )
)

# Avg purchase price (guard division by zero)
purch_agg['AvgPurchasePrice'] = np.where(
    purch_agg['PurchaseQty'] > 0,
    purch_agg['PurchaseDollars'] / purch_agg['PurchaseQty'],
    np.nan
)

purch_agg.head()


,VendorNumber,VendorName,Year,PurchaseQty,PurchaseDollars,AvgPurchasePrice
0,2,"IRA GOLDMAN AND WILLIAMS, LLP",2024,328,5630.88,17.167317
1,54,AAPER ALCOHOL & CHEMICAL CO,2024,1,105.07,105.070000
2,60,ADAMBA IMPORTS INTL INC,2024,4429,71840.44,16.220465
3,60,ADAMBA IMPORTS INTL INC,2025,303,4929.81,16.270000
4,105,ALTAMAR BRANDS LLC,2024,314,11138.18,35.471911


In [4]:
purch_agg.sort_values("PurchaseDollars", ascending=False).head(10)


,VendorNumber,VendorName,Year,PurchaseQty,PurchaseDollars,AvgPurchasePrice
75,3960,DIAGEO NORTH AMERICA INC,2024,5329972,49733333.66,9.330881
79,4425,MARTIGNETTI COMPANIES,2024,2488570,26195391.20,10.526283
179,12546,JIM BEAM BRANDS COMPANY,2024,2610716,22975156.72,8.800328
187,17035,PERNOD RICARD USA,2024,1555833,22866516.19,14.697282
10,480,BACARDI USA INC,2024,1396258,17302147.95,12.391799
27,1392,CONSTELLATION BRANDS INC,2024,2274715,15187741.32,6.676767
20,1128,BROWN-FORMAN CORP,2024,973097,13139921.12,13.503198
150,9165,ULTRA BEVERAGE COMPANY LLP,2024,1038631,12708485.88,12.235805
63,3252,E & J GALLO WINERY,2024,1791181,11858245.15,6.620350
156,9552,M S WALKER INC,2024,1337110,10567196.92,7.903012


In [5]:
purch_agg['VendorNumber'].nunique()


126

## Overall Goal of Sales Aggregation

In [6]:
sales_cols = ['VendorNo', 'VendorName', 'SalesQuantity', 'SalesDollars', 'SalesDate']


In [7]:
sales_chunks = pd.read_csv(
    "../data/sales.csv",
    usecols=sales_cols,
    parse_dates=['SalesDate'],
    chunksize=500_000
)


In [8]:
sales_agg_list = []

for chunk in sales_chunks:
    chunk['Year'] = chunk['SalesDate'].dt.year

    agg = (
        chunk
        .groupby(['VendorNo', 'VendorName', 'Year'], as_index=False)
        .agg(
            SalesQty=('SalesQuantity', 'sum'),
            SalesDollars=('SalesDollars', 'sum')
        )
    )

    sales_agg_list.append(agg)


In [9]:
sales_agg = pd.concat(sales_agg_list, ignore_index=True)


In [10]:
sales_agg = (
    sales_agg
    .groupby(['VendorNo', 'VendorName', 'Year'], as_index=False)
    .agg(
        SalesQty=('SalesQty', 'sum'),
        SalesDollars=('SalesDollars', 'sum')
    )
)


## Compute AvgSalesPrice

In [11]:
sales_agg['AvgSalesPrice'] = np.where(
    sales_agg['SalesQty'] > 0,
    sales_agg['SalesDollars'] / sales_agg['SalesQty'],
    np.nan
)


## 1. How many vendors appear in sales?

In [12]:
sales_agg['VendorNo'].nunique()


127

## 2. Top 10 vendors by SalesDollars

In [13]:
sales_agg.sort_values("SalesDollars", ascending=False).head(10)


,VendorNo,VendorName,Year,SalesQty,SalesDollars,AvgSalesPrice
42,3960,DIAGEO NORTH AMERICA INC,2024,5422703,68742416.99,12.676781
44,4425,MARTIGNETTI COMPANIES,2024,2578355,40992395.93,15.898662
102,17035,PERNOD RICARD USA,2024,1613460,32281247.95,20.007467
98,12546,JIM BEAM BRANDS COMPANY,2024,2630226,31906320.54,12.130638
6,480,BACARDI USA INC,2024,1457301,25014556.89,17.164990
16,1392,CONSTELLATION BRANDS INC,2024,2301688,24469172.93,10.630969
36,3252,E & J GALLO WINERY,2024,1809023,18556085.61,10.257518
12,1128,BROWN-FORMAN CORP,2024,983853,18478557.47,18.781828
82,9165,ULTRA BEVERAGE COMPANY LLP,2024,1043235,17822938.45,17.084299
85,9552,M S WALKER INC,2024,1343481,15465247.75,11.511326


In [14]:
sales_agg.head()


,VendorNo,VendorName,Year,SalesQty,SalesDollars,AvgSalesPrice
0,2,"IRA GOLDMAN AND WILLIAMS, LLP",2024,42,1265.58,30.132857
1,60,ADAMBA IMPORTS INTL INC,2024,3978,67576.22,16.987486
2,105,ALTAMAR BRANDS LLC,2024,325,15748.75,48.457692
3,200,AMERICAN SPIRITS EXCHANGE,2024,103,1719.97,16.698738
4,287,APPOLO VINEYARDS LLC,2024,108,1616.92,14.971481


## STEP 1 — Merge Purchases & Sales on Vendor + Year

In [15]:
vendor_year = pd.merge(
    purch_agg,
    sales_agg,
    left_on=['VendorNumber', 'Year'],
    right_on=['VendorNo', 'Year'],
    how='outer',
    suffixes=('_purch', '_sales')
)


In [16]:
# Prefer vendor name from sales if available, else from purchases
vendor_year['VendorName_final'] = vendor_year['VendorName_sales'].fillna(vendor_year['VendorName_purch'])

# Final vendor number: use VendorNumber (purchases) where not null, else VendorNo
vendor_year['VendorNumber_final'] = vendor_year['VendorNumber'].fillna(vendor_year['VendorNo'])


## STEP 3 — Select and rename final columns

In [17]:
vendor_year = vendor_year[[
    'VendorNumber_final',
    'VendorName_final',
    'Year',
    'PurchaseQty',
    'PurchaseDollars',
    'AvgPurchasePrice',
    'SalesQty',
    'SalesDollars',
    'AvgSalesPrice'
]].copy()

vendor_year.rename(columns={
    'VendorNumber_final': 'VendorNumber',
    'VendorName_final': 'VendorName'
}, inplace=True)


In [18]:
vendor_year['PurchaseDollars'] = vendor_year['PurchaseDollars'].fillna(0)
vendor_year['SalesDollars'] = vendor_year['SalesDollars'].fillna(0)


## NOW COMPUTE PROFIT:

In [19]:
vendor_year['Profit'] = vendor_year['SalesDollars'] - vendor_year['PurchaseDollars']


## MARGIN %:

In [20]:
vendor_year['MarginPct'] = np.where(
    vendor_year['SalesDollars'] > 0,
    vendor_year['Profit'] / vendor_year['SalesDollars'],
    np.nan
)


In [21]:
vendor_year.shape


(247, 11)

## Top vendors by profit

In [22]:
vendor_year.sort_values("Profit", ascending=False).head(10)


,VendorNumber,VendorName,Year,PurchaseQty,PurchaseDollars,AvgPurchasePrice,SalesQty,SalesDollars,AvgSalesPrice,Profit,MarginPct
85,4425.0,MARTIGNETTI COMPANIES,2024,3136.0,40216.11,12.824015,2578355.0,40992395.93,15.898662,40952179.82,0.999019
79,3960.0,DIAGEO NORTH AMERICA INC,2024,5329972.0,49733333.66,9.330881,5422703.0,68742416.99,12.676781,19009083.33,0.276526
83,4425.0,MARTIGNETTI COMPANIES,2024,2488570.0,26195391.20,10.526283,2578355.0,40992395.93,15.898662,14797004.73,0.360970
193,17035.0,PERNOD RICARD USA,2024,1555833.0,22866516.19,14.697282,1613460.0,32281247.95,20.007467,9414731.76,0.291647
27,1392.0,CONSTELLATION BRANDS INC,2024,2274715.0,15187741.32,6.676767,2301688.0,24469172.93,10.630969,9281431.61,0.379311
185,12546.0,JIM BEAM BRANDS COMPANY,2024,2610716.0,22975156.72,8.800328,2630226.0,31906320.54,12.130638,8931163.82,0.279918
10,480.0,BACARDI USA INC,2024,1396258.0,17302147.95,12.391799,1457301.0,25014556.89,17.164990,7712408.94,0.308317
67,3252.0,E & J GALLO WINERY,2024,1791181.0,11858245.15,6.620350,1809023.0,18556085.61,10.257518,6697840.46,0.360951
47,2000.0,SOUTHERN WINE & SPIRITS NE,2024,4198.0,40803.92,9.719848,397696.0,5729494.21,14.406718,5688690.29,0.992878
20,1128.0,BROWN-FORMAN CORP,2024,973097.0,13139921.12,13.503198,983853.0,18478557.47,18.781828,5338636.35,0.288910


## Vendors with negative profit

In [23]:
vendor_year[vendor_year['Profit'] < 0].head(10)


,VendorNumber,VendorName,Year,PurchaseQty,PurchaseDollars,AvgPurchasePrice,SalesQty,SalesDollars,AvgSalesPrice,Profit,MarginPct
0,2.0,"IRA GOLDMAN AND WILLIAMS, LLP",2024,328.0,5630.88,17.167317,42.0,1265.58,30.132857,-4365.30,-3.449249
1,54.0,AAPER ALCOHOL & CHEMICAL CO,2024,1.0,105.07,105.070000,NaN,0.00,NaN,-105.07,NaN
2,60.0,ADAMBA IMPORTS INTL INC,2024,4429.0,71840.44,16.220465,3978.0,67576.22,16.987486,-4264.22,-0.063102
3,60.0,ADAMBA IMPORTS INTL INC,2025,303.0,4929.81,16.270000,NaN,0.00,NaN,-4929.81,NaN
5,105.0,ALTAMAR BRANDS LLC,2025,18.0,568.02,31.556667,NaN,0.00,NaN,-568.02,NaN
7,287.0,APPOLO VINEYARDS LLC,2024,230.0,2399.70,10.433478,108.0,1616.92,14.971481,-782.78,-0.484118
9,388.0,ATLANTIC IMPORTING COMPANY,2025,53.0,1129.96,21.320000,NaN,0.00,NaN,-1129.96,NaN
11,480.0,BACARDI USA INC,2025,30817.0,322230.77,10.456267,NaN,0.00,NaN,-322230.77,NaN
13,516.0,BANFI PRODUCTS CORP,2025,17523.0,124841.19,7.124419,NaN,0.00,NaN,-124841.19,NaN
15,653.0,STATE WINE & SPIRITS,2025,6769.0,70461.85,10.409492,NaN,0.00,NaN,-70461.85,NaN


## Vendors with no sales

In [24]:
vendor_year[(vendor_year['SalesDollars'] == 0)].head(10)


,VendorNumber,VendorName,Year,PurchaseQty,PurchaseDollars,AvgPurchasePrice,SalesQty,SalesDollars,AvgSalesPrice,Profit,MarginPct
1,54.0,AAPER ALCOHOL & CHEMICAL CO,2024,1.0,105.07,105.070000,NaN,0.0,NaN,-105.07,NaN
3,60.0,ADAMBA IMPORTS INTL INC,2025,303.0,4929.81,16.270000,NaN,0.0,NaN,-4929.81,NaN
5,105.0,ALTAMAR BRANDS LLC,2025,18.0,568.02,31.556667,NaN,0.0,NaN,-568.02,NaN
9,388.0,ATLANTIC IMPORTING COMPANY,2025,53.0,1129.96,21.320000,NaN,0.0,NaN,-1129.96,NaN
11,480.0,BACARDI USA INC,2025,30817.0,322230.77,10.456267,NaN,0.0,NaN,-322230.77,NaN
13,516.0,BANFI PRODUCTS CORP,2025,17523.0,124841.19,7.124419,NaN,0.0,NaN,-124841.19,NaN
15,653.0,STATE WINE & SPIRITS,2025,6769.0,70461.85,10.409492,NaN,0.0,NaN,-70461.85,NaN
17,660.0,SAZERAC NORTH AMERICA INC.,2025,22566.0,150099.77,6.651590,NaN,0.0,NaN,-150099.77,NaN
19,1003.0,BRONCO WINE COMPANY,2025,18.0,69.12,3.840000,NaN,0.0,NaN,-69.12,NaN
21,1128.0,BROWN-FORMAN CORP,2025,33025.0,389511.96,11.794458,NaN,0.0,NaN,-389511.96,NaN


In [26]:
vendor_year.to_csv("../data_processed/vendor_year_metrics.csv", index=False)
